In [15]:
# Core packages
import os  # for file and path operations
import warnings  # to suppress warnings
warnings.filterwarnings("ignore")

# Data manipulation
import pandas as pd  # for handling dataframes
import numpy as np  # for numerical operations

# Visualization
import matplotlib.pyplot as plt  # for general plotting
import seaborn as sns  # for enhanced statistical plotting
from matplotlib.pylab import rcParams  # for customizing plot size and appearance

# Machine learning - preprocessing & model evaluation
from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score  # data splitting and cross-validation
from sklearn.preprocessing import StandardScaler  # feature standardization

# Machine learning - models
from sklearn.tree import DecisionTreeClassifier, plot_tree  # decision tree model and plotting
from sklearn.ensemble import RandomForestClassifier  # random forest model

# Machine learning - metrics
from sklearn.metrics import (
    ConfusionMatrixDisplay,  # plot confusion matrix
    classification_report,  # detailed classification report
    confusion_matrix,  # confusion matrix array
    accuracy_score,  # accuracy metric
    precision_score,  # precision metric
    recall_score,  # recall metric
    f1_score  # F1 score
)

# PyTorch - deep learning
import torch  # base PyTorch package
import torch.nn as nn  # neural network layers
import torch.optim as optim  # optimizers (e.g., Adam, SGD)

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

In [16]:
# Load data
full = pd.read_csv("C:/Users/annaw/Desktop/DataScience/Datasets/full_dataset.csv", dtype={91: str})
print(f"X_train shape: {full.shape}")

# Drop columns with >20% missing
full_clean = full.dropna(thresh=len(full)*0.8, axis=1)  
print(full_clean['avalancheDay1'].value_counts())

# Save year column before dropping others
years = full_clean['year']
print(years)
# Drop rows with any missing values
full_clean = full_clean.dropna()

# Select only numeric columns
full_clean = full_clean.select_dtypes(include=[np.number])

# Drop unwanted columns (keep year for now)
if 'datum' in full_clean.columns:
    full_clean = full_clean.drop(columns=['datum'])

# Add year back to use for splitting
full_clean['year'] = years.loc[full_clean.index]

# Define features and target
X = full_clean.drop(columns=["avalancheDay1"])
y = full_clean["avalancheDay1"]

# Split using year == 22 for val/test, rest for training
X_train = X[X['year'] != 2022].drop(columns=['year'])
y_train = y[X['year'] != 2022]

X_val_test = X[X['year'] == 2022].drop(columns=['year'])
y_val_test = y[X['year'] == 2022]

# Optional: split year 22 set further into val and test
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, test_size=0.5, random_state=42)

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Save for SHAP
X_background = X_train_scaled[:100]
X_explain = X_test_scaled

# Convert to PyTorch tensors
X_train_nn = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_nn = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_nn = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_val = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)


X_train shape: (11362, 95)
avalancheDay1
0    10638
1      724
Name: count, dtype: int64
0        2020
1        2020
2        2020
3        2020
4        2020
         ... 
11357    2022
11358    2021
11359    2022
11360    2022
11361    2022
Name: year, Length: 11362, dtype: int64


In [26]:
#Wrap the pytorch model for stacking
class TorchNNWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, input_dim, threshold=0.5, epochs=100, lr=0.001, pos_weight=5.0):
        self.threshold = threshold
        self.epochs = epochs
        self.lr = lr
        self.pos_weight = pos_weight  
        self.input_dim = input_dim

    def _build_model(self):
        model = nn.Sequential(
            nn.Linear(self.input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )
        return model

    def fit(self, X, y):
        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

        self.model = self._build_model()
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([self.pos_weight]))

        for _ in range(self.epochs):
            self.model.train()
            optimizer.zero_grad()
            outputs = self.model(X_tensor)
            loss = criterion(outputs, y_tensor)
            loss.backward()
            optimizer.step()
            
        self.classes_ = np.array([0, 1])  # binary classification
        return self

    def predict_proba(self, X):
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.tensor(X, dtype=torch.float32)
            logits = self.model(X_tensor)
            probs = torch.sigmoid(logits).numpy().flatten()
        return np.vstack([1 - probs, probs]).T

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > self.threshold).astype(int)

In [27]:
# Wrap PyTorch model
nn_wrapper = TorchNNWrapper(input_dim=X_train_scaled.shape[1], threshold=0.5, epochs=200)

In [28]:
#F1 optimized RF
rf_custom_F1 = RandomForestClassifier(
    class_weight=None,
    max_depth=20,
    max_features=None,
    min_samples_leaf=5,
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

#Recall optimized RF

rf_custom_Recall = RandomForestClassifier(
    class_weight='balanced',
    max_depth=10,
    max_features="sqrt",
    min_samples_leaf=200,
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [29]:
# Build stacking ensemble
stacked_model = StackingClassifier(
    estimators=[
        ('rf_f1', rf_custom_F1),
        ('rf_recall', rf_custom_Recall),
        ('nn', nn_wrapper)
    ],
    final_estimator=LogisticRegression(),  # Can also use RF or Gradient Boosting here
    cv=5,  # Cross-validation for meta-learner
    passthrough=False  # If True, pass original features along with base learner outputs
)

# Fit ensemble model
stacked_model.fit(X_train_scaled, y_train)

# Predict
y_pred_stack = stacked_model.predict(X_val_scaled)
print("\n=== Stacked Model Performance ===")
print(classification_report(y_val, y_pred_stack))
print(confusion_matrix(y_val, y_pred_stack))



=== Stacked Model Performance ===
              precision    recall  f1-score   support

         0.0       1.00      0.99      0.99       907
         1.0       0.69      0.91      0.78        32

    accuracy                           0.98       939
   macro avg       0.84      0.95      0.89       939
weighted avg       0.99      0.98      0.98       939

[[894  13]
 [  3  29]]


In [34]:
# Build stacking ensemble
stacked_model = StackingClassifier(
    estimators=[
        ('rf_f1', rf_custom_F1),
        ('rf_recall', rf_custom_Recall),
        ('nn', nn_wrapper)
    ],
    final_estimator=LogisticRegression(),  # Can also use RF or Gradient Boosting here
    cv=5,  # Cross-validation for meta-learner
    passthrough=True  # If True, pass original features along with base learner outputs
)

# Fit ensemble model
stacked_model.fit(X_train_scaled, y_train)

# Predict
y_pred_stack = stacked_model.predict(X_val_scaled)
print("\n=== Stacked Model Performance ===")
print(classification_report(y_val, y_pred_stack))
print(confusion_matrix(y_val, y_pred_stack))



=== Stacked Model Performance ===
              precision    recall  f1-score   support

         0.0       1.00      0.99      0.99       907
         1.0       0.71      0.91      0.79        32

    accuracy                           0.98       939
   macro avg       0.85      0.95      0.89       939
weighted avg       0.99      0.98      0.98       939

[[895  12]
 [  3  29]]


In [35]:
from sklearn.ensemble import GradientBoostingClassifier

stacked_model = StackingClassifier(
    estimators=[
        ('rf_f1', rf_custom_F1),
        ('rf_recall', rf_custom_Recall),
        ('nn', nn_wrapper)
    ],
    final_estimator=GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3),
    cv=5,
    passthrough=False
)

# Fit ensemble model
stacked_model.fit(X_train_scaled, y_train)

# Predict
y_pred_stack = stacked_model.predict(X_val_scaled)
print("\n=== Stacked Model Performance ===")
print(classification_report(y_val, y_pred_stack))
print(confusion_matrix(y_val, y_pred_stack))



=== Stacked Model Performance ===
              precision    recall  f1-score   support

         0.0       1.00      0.99      0.99       907
         1.0       0.68      0.88      0.77        32

    accuracy                           0.98       939
   macro avg       0.84      0.93      0.88       939
weighted avg       0.98      0.98      0.98       939

[[894  13]
 [  4  28]]


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

stacked_model = StackingClassifier(
    estimators=[
        ('rf_f1', rf_custom_F1),
        ('rf_recall', rf_custom_Recall),
        ('nn', nn_wrapper)
    ],
    final_estimator=GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3),
    cv=5,
    passthrough=True
)

# Fit ensemble model
stacked_model.fit(X_train_scaled, y_train)

# Predict
y_pred_stack = stacked_model.predict(X_val_scaled)
print("\n=== Stacked Model Performance ===")
print(classification_report(y_val, y_pred_stack))
print(confusion_matrix(y_val, y_pred_stack))

#It gets super duper good if i use passthrough = True 
#This gives the meta-learner access to the original features and the predictions